# Assignment 5: RAG Agent

**Goal:** Build a Retrieval-Augmented Generation (RAG) agent that:
1. **Loads** a blog post about Prompt Engineering (Lilian Weng)
2. **Splits & embeds** the content into a vector store
3. **Creates a ReAct agent** with a `retrieve` tool
4. The agent can **search the blog post** to answer questions accurately

**Data source:** Different from Assignment 4 - uses a web blog post instead of Vision 2030 PDF

## 1. Setup & Imports

In [1]:
import os
import bs4
from dotenv import load_dotenv
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

load_dotenv(os.path.join(os.getcwd(), '..', '.env'))

model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

print("Model initialized successfully")

USER_AGENT environment variable not set, consider setting it to identify your requests.


ModuleNotFoundError: No module named 'langchain_huggingface'

## 2. Load Blog Post from Web

In [2]:
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()
print(f"Loaded {len(docs)} document(s) from blog post")
print(f"Content length: {len(docs[0].page_content)} characters")

Loaded 1 document(s) from blog post
Content length: 29286 characters


## 3. Split & Embed

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)
print(f"Split into {len(all_splits)} chunks")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

NameError: name 'RecursiveCharacterTextSplitter' is not defined

## 4. Store in Vector Store

In [4]:
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(documents=all_splits)
print("Stored all chunks in vector store")

NameError: name 'embeddings' is not defined

## 5. RAG Agent with Retrieve Tool

In [5]:
@tool
def retrieve(query: str) -> str:
    """Search the prompt engineering blog post for relevant information."""
    retrieved_docs = vector_store.similarity_search(query, k=3)
    return "\n\n".join(
        f"Source: {doc.metadata}\nContent: {doc.page_content}"
        for doc in retrieved_docs
    )

agent = create_react_agent(
    model=model,
    tools=[retrieve],
    prompt=(
        "You are a helpful assistant that answers questions about prompt engineering techniques. "
        "You have a retrieve tool that searches a blog post for relevant info. "
        "Use it to answer questions. You can call it multiple times with different queries if needed."
    ),
)

print("RAG agent created with retrieve tool")

NameError: name 'create_react_agent' is not defined

## 6. Demo: Simple Question

In [6]:
print("=" * 70)
print("Question: What is few-shot prompting?")
print("=" * 70)

result = agent.invoke({
    "messages": [HumanMessage(content="What is few-shot prompting?")]
})
print(f"\nAnswer: {result['messages'][-1].content}")

Question: What is few-shot prompting?


NameError: name 'agent' is not defined

## 7. Demo: Multi-step Question

In [7]:
print("=" * 70)
print("Question: What is Chain of Thought prompting? And how does it compare to zero-shot prompting?")
print("=" * 70)

result = agent.invoke({
    "messages": [HumanMessage(
        content="What is Chain of Thought prompting? And how does it compare to zero-shot prompting?"
    )]
})
print(f"\nAnswer: {result['messages'][-1].content}")

Question: What is Chain of Thought prompting? And how does it compare to zero-shot prompting?


NameError: name 'agent' is not defined